# Cluster Composition Mixing Entropy

This notebook tests whether sociodemographic mixing within clusters predicts `large_cluster` versus `small_cluster` among non-singleton clusters.

It runs two membership definitions:

1. `all_windows`: all primary-resolution cluster-window memberships are retained. This is the main cluster-composition analysis because the cluster composition is measured on the full cluster-window object.
2. `largest_cluster_nonoverlap`: each sequence is assigned to one cluster across overlapping windows by choosing the largest cluster it appears in, with ties broken by closeness to the window midpoint and then earlier window index.

Mixing is measured with normalised Shannon entropy for age, sex, SIMD quintile, age x sex demographic profile, and age x sex x SIMD sociodemographic profile. Entropy is normalised by `log(min(K, n_valid))`, which keeps the maximum at 1 for small clusters that cannot contain every global category.

To avoid fitting highly overlapping component and joint entropy variables in the same random forest, each membership definition fits three separated model families: individual mixing (`age + sex + SIMD`), demographic mixing plus SIMD (`age x sex + SIMD`), and full sociodemographic mixing only (`age x sex x SIMD`). Each family is fit with entropy-only and context-adjusted predictors.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'cluster_size_rf').exists():
    PROJECT_ROOT = Path('/Users/ydnkka/Desktop/PhD Project/projects/scotland')

os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib'))
(PROJECT_ROOT / '.matplotlib').mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from cluster_size_rf.lib.config import AnalysisConfig, OUT_DIR
from cluster_size_rf.lib.data import (
    prepare_cluster_membership_inputs,
    prepare_model_inputs,
    read_analysis_data,
)
from cluster_size_rf.lib.mixing_entropy import run_mixing_entropy_workflow


## Configuration

In [2]:
RUN_MODE = 'quick'  # change to 'final' for the full run

config = AnalysisConfig.from_run_mode(
    run_mode=RUN_MODE,
    primary_resolution=0.3,
    primary_large_min=13,
    window_dedup_strategy='largest_cluster',
    simd_overall_mode='quintile',
    simd_domain_mode='quintile',
    simd_band_weighting='population',
    save_models=True,
)

MIXING_OUT_DIR = OUT_DIR / 'mixing_entropy'
MIXING_OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.Series(config.to_dict())

run_mode                                           quick
primary_resolution                                   0.3
primary_large_min                                     13
random_state                                          42
window_dedup_strategy                    largest_cluster
simd_overall_mode                               quintile
simd_domain_mode                                quintile
simd_band_weighting                           population
use_sensitivity_context_controls                   False
fit_singleton_binary                                True
fit_simd_domain_decomposition                       True
fit_secondary_multiclass                            True
run_threshold_sensitivity                          False
run_resolution_sensitivity                         False
sensitivity_large_mins                          (10, 20)
sensitivity_resolutions             (0.1, 0.2, 0.4, 0.5)
save_models                                         True
n_estimators                   

## Prepare Membership Tables

In [3]:
raw = read_analysis_data()

# Main cluster-composition dataset: all cluster-window memberships.
all_windows_df, all_windows_feature_spec = prepare_cluster_membership_inputs(raw, config)

# Non-overlapping sequence-level sensitivity: one cluster per sequence, chosen by largest cluster size.
largest_nonoverlap_df, _, largest_nonoverlap_feature_spec, _ = prepare_model_inputs(raw, config)

prep_summary = pd.DataFrame(
    [
        {
            'analysis': 'all_windows',
            'rows': len(all_windows_df),
            'unique_sequences': all_windows_df['sequence_id'].nunique(),
            'unique_cluster_ids': all_windows_df['cluster_id'].nunique(),
            'simd_feature': all_windows_feature_spec.simd_overall_feature,
        },
        {
            'analysis': 'largest_cluster_nonoverlap',
            'rows': len(largest_nonoverlap_df),
            'unique_sequences': largest_nonoverlap_df['sequence_id'].nunique(),
            'unique_cluster_ids': largest_nonoverlap_df['cluster_id'].nunique(),
            'simd_feature': largest_nonoverlap_feature_spec.simd_overall_feature,
        },
    ]
)
prep_summary.to_csv(MIXING_OUT_DIR / 'cluster_mixing_membership_prep_summary.csv', index=False)
display(prep_summary)

,analysis,rows,unique_sequences,unique_cluster_ids,simd_feature
0,all_windows,851442,303239,206736,dz_simd_pop_quintile
1,largest_cluster_nonoverlap,303239,303239,79299,dz_simd_pop_quintile


## Run Models And Save Outputs

In [4]:
analysis_inputs = {
    'all_windows': {
        'model_df': all_windows_df,
        'feature_spec': all_windows_feature_spec,
        'membership_definition': 'All primary-resolution cluster-window memberships; sequences may contribute to multiple overlapping windows.',
    },
    'largest_cluster_nonoverlap': {
        'model_df': largest_nonoverlap_df,
        'feature_spec': largest_nonoverlap_feature_spec,
        'membership_definition': 'One row per sequence; overlapping-window membership chosen by largest cluster size, then closest midpoint.',
    },
}

workflow_outputs = {}
for label, spec in analysis_inputs.items():
    print(f'Running {label}...')
    workflow_outputs[label] = run_mixing_entropy_workflow(
        label=label,
        model_df=spec['model_df'],
        feature_spec=spec['feature_spec'],
        config=config,
        out_dir=MIXING_OUT_DIR / label,
        membership_definition=spec['membership_definition'],
        save_models=config.save_models,
    )

combined_metrics = pd.concat(
    [out['metrics'] for out in workflow_outputs.values()],
    ignore_index=True,
)
combined_counts = pd.concat(
    [
        out['cluster_counts'].assign(analysis=label)
        for label, out in workflow_outputs.items()
    ],
    ignore_index=True,
)
combined_entropy_summary = pd.concat(
    [
        out['entropy_summary'].assign(analysis=label)
        for label, out in workflow_outputs.items()
    ],
    ignore_index=True,
)

combined_metrics.to_csv(MIXING_OUT_DIR / 'cluster_mixing_membership_comparison_metrics.csv', index=False)
combined_counts.to_csv(MIXING_OUT_DIR / 'cluster_mixing_membership_comparison_counts.csv', index=False)
combined_entropy_summary.to_csv(MIXING_OUT_DIR / 'cluster_mixing_membership_comparison_entropy_summary.csv', index=False)

display(combined_counts)
display(combined_metrics)

Running all_windows...
Running largest_cluster_nonoverlap...


,cluster_type,n_clusters,prop_clusters,analysis
0,small_cluster,85839,0.895584,all_windows
1,large_cluster,10008,0.104416,all_windows
2,small_cluster,41131,0.844336,largest_cluster_nonoverlap
3,large_cluster,7583,0.155664,largest_cluster_nonoverlap


,analysis,membership_definition,model,splitter,label,n_rows,balanced_accuracy,macro_f1,roc_auc,average_precision
0,all_windows,All primary-resolution cluster-window membersh...,individual_entropy,StratifiedGroupKFold(n_splits=5),all_windows_individual_entropy_holdout,19166,0.944690,0.839631,0.987944,0.918903
1,all_windows,All primary-resolution cluster-window membersh...,individual_context_adjusted,StratifiedGroupKFold(n_splits=5),all_windows_individual_context_adjusted_holdout,19166,0.931481,0.827504,0.980945,0.871075
2,all_windows,All primary-resolution cluster-window membersh...,demographic_simd_entropy,StratifiedGroupKFold(n_splits=5),all_windows_demographic_simd_entropy_holdout,19166,0.955207,0.866796,0.991090,0.937969
3,all_windows,All primary-resolution cluster-window membersh...,demographic_simd_context_adjusted,StratifiedGroupKFold(n_splits=5),all_windows_demographic_simd_context_adjusted_...,19166,0.927667,0.823633,0.980548,0.862693
4,all_windows,All primary-resolution cluster-window membersh...,sociodemographic_entropy,StratifiedGroupKFold(n_splits=5),all_windows_sociodemographic_entropy_holdout,19166,0.924848,0.923632,0.943765,0.883314
5,all_windows,All primary-resolution cluster-window membersh...,sociodemographic_context_adjusted,StratifiedGroupKFold(n_splits=5),all_windows_sociodemographic_context_adjusted_...,19166,0.896638,0.828557,0.958669,0.803649
6,largest_cluster_nonoverlap,One row per sequence; overlapping-window membe...,individual_entropy,StratifiedGroupKFold(n_splits=5),largest_cluster_nonoverlap_individual_entropy_...,9743,0.787359,0.749402,0.840567,0.687657
7,largest_cluster_nonoverlap,One row per sequence; overlapping-window membe...,individual_context_adjusted,StratifiedGroupKFold(n_splits=5),largest_cluster_nonoverlap_individual_context_...,9743,0.789142,0.746724,0.868188,0.701363
8,largest_cluster_nonoverlap,One row per sequence; overlapping-window membe...,demographic_simd_entropy,StratifiedGroupKFold(n_splits=5),largest_cluster_nonoverlap_demographic_simd_en...,9743,0.789193,0.764885,0.836546,0.680373
9,largest_cluster_nonoverlap,One row per sequence; overlapping-window membe...,demographic_simd_context_adjusted,StratifiedGroupKFold(n_splits=5),largest_cluster_nonoverlap_demographic_simd_co...,9743,0.788467,0.747458,0.866230,0.697375


## Compare Feature Importance

In [5]:
importance_tables = []
for label in analysis_inputs:
    for path in sorted((MIXING_OUT_DIR / label).glob(f'{label}_*_context_adjusted_permutation_importance.csv')):
        model = path.name.removeprefix(f'{label}_').removesuffix('_permutation_importance.csv')
        imp = pd.read_csv(path).assign(analysis=label, model=model)
        importance_tables.append(imp)

combined_importance = pd.concat(importance_tables, ignore_index=True)
combined_importance.to_csv(MIXING_OUT_DIR / 'cluster_mixing_membership_comparison_context_adjusted_importance.csv', index=False)

display(
    combined_importance
    .sort_values(['analysis', 'model', 'mean_drop_average_precision'], ascending=[True, True, False])
    .groupby(['analysis', 'model'])
    .head(8)
)

,feature,mean_drop_average_precision,std_drop_average_precision,analysis,model
0,demographic_entropy_norm,0.475808,0.009167,all_windows,demographic_simd_context_adjusted
1,simd_entropy_norm,0.427783,0.006190,all_windows,demographic_simd_context_adjusted
2,log_dz_population_density_mean,0.056473,0.005581,all_windows,demographic_simd_context_adjusted
3,dz_cum_prop_sequenced_mean,0.044851,0.002722,all_windows,demographic_simd_context_adjusted
4,cluster_mid_days_since_epoch,0.032414,0.003426,all_windows,demographic_simd_context_adjusted
5,log_dz_cum_incidence_per_capita_mean,0.031524,0.003829,all_windows,demographic_simd_context_adjusted
6,dz_7d_test_positivity_mean,0.027595,0.001881,all_windows,demographic_simd_context_adjusted
7,wn_prop_sequenced,0.015347,0.000665,all_windows,demographic_simd_context_adjusted
10,simd_entropy_norm,0.453638,0.007786,all_windows,individual_context_adjusted
11,age_entropy_norm,0.342618,0.009033,all_windows,individual_context_adjusted


## Saved Output Locations

In [6]:
for label in analysis_inputs:
    print(label, MIXING_OUT_DIR / label)
print('Combined comparison tables:', MIXING_OUT_DIR)

all_windows /Users/ydnkka/Desktop/PhD Project/projects/scotland/cluster_size_rf/outputs/mixing_entropy/all_windows
largest_cluster_nonoverlap /Users/ydnkka/Desktop/PhD Project/projects/scotland/cluster_size_rf/outputs/mixing_entropy/largest_cluster_nonoverlap
Combined comparison tables: /Users/ydnkka/Desktop/PhD Project/projects/scotland/cluster_size_rf/outputs/mixing_entropy
